# ONNX Runtime Custom Op: Conv_ReLU6

This notebook focuses on:
1. Loading fused model with ORT custom op library.
2. Consistency check and latency benchmark in ONNX Runtime.


## Setup


In [1]:
import os
import time
import pathlib
import numpy as np
import onnxruntime as ort


## Paths


In [2]:
orig_model_path = 'data/public/mobilenetv2-12/mobilenetv2-12.onnx'
fused_model_path = 'data/public/mobilenetv2-12/mobilenetv2_fused_full.onnx'
custom_ops_library = r'ort_custom_ops/build/Release/ort_custom_ops.dll'
input_shape = [1, 3, 224, 224]

print('onnxruntime version:', ort.__version__)
print('onnxruntime package:', ort.__file__)


onnxruntime version: 1.23.2
onnxruntime package: d:\projects\conda_envs\llama\lib\site-packages\onnxruntime\__init__.py


In [ ]:
# Build custom op, refer to conv_relu6_custom_op.cc.

## Build ORT Session Helpers


In [3]:
def benchmark_ort_session(sess, input_shape, warmup=10, runs=50, seed=42):
    input_name = sess.get_inputs()[0].name
    np.random.seed(seed)
    x = np.random.randn(*input_shape).astype(np.float32)

    for _ in range(warmup):
        _ = sess.run(None, {input_name: x})

    costs = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _ = sess.run(None, {input_name: x})
        costs.append((time.perf_counter() - t0) * 1000.0)

    arr = np.array(costs, dtype=np.float64)
    return {
        'mean_ms': float(arr.mean()),
        'median_ms': float(np.median(arr)),
        'p95_ms': float(np.percentile(arr, 95)),
        'min_ms': float(arr.min()),
        'max_ms': float(arr.max()),
    }


def make_ort_session(model_path, custom_op_lib=None, providers=None):
    if providers is None:
        providers = ['CPUExecutionProvider']

    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    if custom_op_lib:
        if not os.path.exists(custom_op_lib):
            raise FileNotFoundError(f'custom op library not found: {custom_op_lib}')

        # Ensure dependent onnxruntime.dll can be found on Windows.
        ort_pkg_dir = pathlib.Path(ort.__file__).resolve().parent
        capi_dir = ort_pkg_dir / 'capi'
        if hasattr(os, 'add_dll_directory'):
            os.add_dll_directory(str(capi_dir))
        os.environ['PATH'] = str(capi_dir) + os.pathsep + os.environ.get('PATH', '')

        so.register_custom_ops_library(custom_op_lib)

    return ort.InferenceSession(model_path, sess_options=so, providers=providers)


## Benchmark: Original vs Fused


In [4]:
print('=== ORT Benchmark ===')
print('Original model:', orig_model_path)
orig_sess = make_ort_session(orig_model_path)
orig_stat = benchmark_ort_session(orig_sess, input_shape)
print('Original:', orig_stat)

print('Fused model:', fused_model_path)
fused_sess = make_ort_session(fused_model_path, custom_op_lib=custom_ops_library)
fused_stat = benchmark_ort_session(fused_sess, input_shape)
print('Fused:', fused_stat)
print(f"Speedup (orig/fused, mean): {orig_stat['mean_ms'] / fused_stat['mean_ms']:.4f}x")


=== ORT Benchmark ===
Original model: data/public/mobilenetv2-12/mobilenetv2-12.onnx
Original: {'mean_ms': 2.8781380061991513, 'median_ms': 2.781549992505461, 'p95_ms': 3.4989450185094024, 'min_ms': 2.311199903488159, 'max_ms': 5.035900045186281}
Fused model: data/public/mobilenetv2-12/mobilenetv2_fused_full.onnx
Fused: {'mean_ms': 724.3550300085917, 'median_ms': 725.885899970308, 'p95_ms': 756.6790249838959, 'min_ms': 668.6985000269488, 'max_ms': 777.0001000026241}
Speedup (orig/fused, mean): 0.0040x


## Consistency Check in ORT


In [5]:
x = np.random.randn(*input_shape).astype(np.float32)
orig_input = orig_sess.get_inputs()[0].name
fused_input = fused_sess.get_inputs()[0].name

y0 = orig_sess.run(None, {orig_input: x})[0]
y1 = fused_sess.run(None, {fused_input: x})[0]

print('allclose:', np.allclose(y0, y1, rtol=1e-5, atol=1e-5))
print('max abs diff:', float(np.max(np.abs(y0 - y1))))


allclose: True
max abs diff: 4.291534423828125e-06


## Common Issues and Fixes

### 1) Version conflict with stale `onnxruntime.dll` in `System32`
**Symptom**:
- Kernel crash / API mismatch, e.g. requested API version not available.

**Fix used in practice**:
- Remove/replace outdated `onnxruntime.dll` under `System32`.
- Ensure only one ORT runtime version is resolved by process search path.
- Keep Python `onnxruntime` and custom-op build ORT version aligned.

### 2) Custom op library load fails (`RegisterCustomOps` not found)
**Fix**:
- Export symbol explicitly in C++:
  - `extern "C" __declspec(dllexport) OrtStatus* ORT_API_CALL RegisterCustomOps(...)`

### 3) `onnxruntime.dll` not found when loading custom op DLL
**Fix**:
- Add ORT package `capi` folder to DLL search path before `register_custom_ops_library`.


## QA

### Q1: Why is fused model much slower with custom op than original ORT model?
- Custom kernel is a naive CPU implementation.
- Original ORT Conv uses highly optimized kernels.

### Q2: Does custom op always improve performance?
- No. Functional correctness and deployment flexibility may improve first.
- Performance needs optimized kernel implementation.


## Summary

This notebook demonstrated:
1. ORT custom op loading for `Conv_ReLU6`.
2. ORT-side consistency and latency comparison.
3. Practical troubleshooting for version and DLL conflicts.
